# A units + dimensional-analysis + error-propagation calculator

[`dgs/dimensional_analysis.py`](../dgs/dimensional_analysis.py) checks
whether two SymPy unit expressions reduce to the same base SI dimensions.
[`dgs/error_propagation.py`](../dgs/error_propagation.py) propagates
uncertainty through arithmetic (add-in-quadrature, relative-quadrature,
$|n|$-scaling). Neither module knows the other exists — a `Measurement` has
no unit, and `dims_equal` has no uncertainty.
[`dgs/units_error_calculator.py`](../dgs/units_error_calculator.py)'s
`DimensionalMeasurement` is the missing combination: every `+`/`-` checks
dimensional compatibility, every operation propagates $\sigma$ using
`error_propagation`'s own rules (reused, not re-derived).

This notebook is honest about scope: it catches genuine **dimensional**
mismatches (force + energy), not unit-**system** conversion-factor bugs
(newtons vs. pound-force) — those are dimensionally identical and a
different kind of mistake, the actual Mars Climate Orbiter failure mode.


In [1]:
import sys, pathlib
from sympy.physics import units as u

REPO = pathlib.Path(r"D:/Summer2026/Dispersion-Assisted-GS-Phase-Recovery")
sys.path.insert(0, str(REPO))
from dgs import units_error_calculator as uec
from dgs import dimensional_analysis as da
from dgs import error_propagation as ep

checks = []


def check(label, condition):
    checks.append((label, bool(condition)))
    print(f"{'PASS' if condition else 'FAIL'}  —  {label}")


## 1. Griffiths 7.13: $\text{emf} = Bhv$, with real units and real uncertainty

Same numbers `error_propagation.py`'s own demo uses — this time every
quantity carries its unit, and the result is checked to actually *be* a
Volt, not just numerically resemble one.


In [2]:
B = uec.DimensionalMeasurement(0.5, 0.01, u.tesla)
h = uec.DimensionalMeasurement(2.0, 0.05, u.meters)
v = uec.DimensionalMeasurement(3.0, 0.1, u.meters / u.seconds)

emf = B * h * v
print(f"emf = B*h*v = {emf}")
print(f"is emf's unit dimensionally a Volt? {emf.convert_check(u.volts)}")

val_ref, sig_ref = ep.propagate(lambda p: p[0] * p[1] * p[2], [0.5, 2.0, 3.0], [0.01, 0.05, 0.1])
print(f"cross-check vs error_propagation.propagate() (unit-free): {val_ref:.4f} +/- {sig_ref:.5f}")

check("emf's propagated value/sigma match error_propagation.propagate() exactly",
      abs(emf.value - val_ref) < 1e-9 and abs(emf.sigma - sig_ref) < 1e-9)
check("emf's resulting unit is dimensionally a Volt", emf.convert_check(u.volts))


emf = B*h*v = 3 +/- 0.139  [meter**2*tesla/second]
is emf's unit dimensionally a Volt? True
cross-check vs error_propagation.propagate() (unit-free): 3.0000 +/- 0.13865
PASS  —  emf's propagated value/sigma match error_propagation.propagate() exactly
PASS  —  emf's resulting unit is dimensionally a Volt


## 2. Catching a real dimensional mismatch

Force and energy are not the same dimension ($[F]=ML/T^2$,
$[E]=ML^2/T^2$) — adding them must be rejected, not silently produce a
wrong number.


In [3]:
force = uec.DimensionalMeasurement(10.0, 0.2, u.newtons)
energy = uec.DimensionalMeasurement(5.0, 0.1, u.joules)

try:
    bad = force + energy
    print("BUG: should have been rejected:", bad)
    caught = False
except ValueError as e:
    print(f"Correctly rejected: {e}")
    caught = True

check("force + energy is rejected as a dimensional mismatch", caught)

# same dimension (two forces) DOES combine, with sigma adding in quadrature
f1 = uec.DimensionalMeasurement(10.0, 0.2, u.newtons)
f2 = uec.DimensionalMeasurement(3.0, 0.1, u.newtons)
total = f1 + f2
print(f"\n{f1} + {f2} = {total}")
check("two forces add correctly, sigma adds in quadrature",
      abs(total.value - 13.0) < 1e-9 and abs(total.sigma - ep.add_in_quadrature(0.2, 0.1)) < 1e-9)


Correctly rejected: cannot add incompatible units: newton vs joule -- these do not reduce to the same base SI dimensions.
PASS  —  force + energy is rejected as a dimensional mismatch

10 +/- 0.2  [newton] + 3 +/- 0.1  [newton] = 13 +/- 0.224  [newton]
PASS  —  two forces add correctly, sigma adds in quadrature


## 3. The power rule, with units

$(10\pm0.2\text{ m})^2$: area, and the relative uncertainty doubles
(power rule with $n=2$).


In [4]:
length = uec.DimensionalMeasurement(10.0, 0.2, u.meters)
area = length ** 2
print(f"({length})^2 = {area}")

expected_sigma = ep.power_rule(100.0, 10.0, 0.2, 2)
check("Area's value and sigma match error_propagation.power_rule",
      abs(area.value - 100.0) < 1e-9 and abs(area.sigma - expected_sigma) < 1e-9)
check("Area's unit is dimensionally m^2", da.dims_equal(area.unit, u.meters ** 2))


(10 +/- 0.2  [meter])^2 = 100 +/- 4  [meter**2]
PASS  —  Area's value and sigma match error_propagation.power_rule
PASS  —  Area's unit is dimensionally m^2


## 4. What this calculator honestly does NOT catch

Newton-seconds and pound-force-seconds are the *same* dimension
(force$\times$time) — `dims_equal` correctly says they match, even though
$1\ \text{N} \ne 1\ \text{lbf}$ numerically. This is precisely the failure
mode that lost the Mars Climate Orbiter
(`dimensional_analysis.mars_climate_orbiter_case_study`): a unit-*system*
conversion-factor error, not a dimensional one. Catching that needs a
numeric conversion check (SymPy's `convert_to`), which is a fundamentally
different job from comparing dimensional dependencies.


In [5]:
impulse_N_s = u.newtons * u.seconds
impulse_lbf_s = u.pounds * u.acceleration_due_to_gravity * u.seconds

same_dim = da.dims_equal(impulse_N_s, impulse_lbf_s)
print(f"N*s vs (lbf-equivalent)*s dims_equal: {same_dim}  (both are force*time -- correctly the SAME dimension)")
print("This calculator would NOT reject adding these, even though the numeric")
print("values are NOT interchangeable without a conversion factor -- exactly")
print("the class of bug dims_equal is not designed to catch.")

check("N*s and lbf-equivalent*s are correctly recognized as the SAME dimension",
      same_dim is True)


N*s vs (lbf-equivalent)*s dims_equal: True  (both are force*time -- correctly the SAME dimension)
This calculator would NOT reject adding these, even though the numeric
values are NOT interchangeable without a conversion factor -- exactly
the class of bug dims_equal is not designed to catch.
PASS  —  N*s and lbf-equivalent*s are correctly recognized as the SAME dimension


## Final grade

In [6]:
failures = [label for label, ok in checks if not ok]
print(f"{len(checks) - len(failures)}/{len(checks)} checks passed")

if failures:
    raise AssertionError("Failed checks: " + ", ".join(failures))
else:
    print("\nALL CHECKS PASSED — DimensionalMeasurement propagates uncertainty exactly "
          "like error_propagation's own rules, catches real dimensional mismatches, and "
          "is explicit about the one class of bug (unit-system conversion factors) it "
          "cannot and does not claim to catch.")


7/7 checks passed

ALL CHECKS PASSED — DimensionalMeasurement propagates uncertainty exactly like error_propagation's own rules, catches real dimensional mismatches, and is explicit about the one class of bug (unit-system conversion factors) it cannot and does not claim to catch.
